### Prepare environment

In [0]:
%run ../environment/prepare_environment

# Transfer Learning

## Introduction
This notebook demonstrates why Transfer Learning is the industry standard for computer vision tasks, especially when compute resources and labeled data are limited. We compare a from-scratch Multi-Layer Perceptron (MLP) against a pre-trained MobileNetV2 backbone trained on ImageNet.

This notebook will cover:
- Why transfer learning is a practical approach for vision tasks
- Comparing from-scratch training vs. using pre-trained models
- Freezing and fine-tuning strategies
- Using MLflow to compare training efficiency
- Building production-ready models on limited budgets

## 1. Load CIFAR-10 subset

The CIFAR-10 dataset has 60,000 32x32 RGB images across 10 object classes (airplanes, cars, birds, etc.). Unlike MNIST (digits only), CIFAR is more complicated task.

We use subsets (20K train, 4K test) to simulate real-world scenarios where labeled data is expensive. Transfer learning is crucial here: training a million-parameter model from scratch on 20K images would be sample-inefficient and prone to overfitting.

In [0]:
import os
import time
import random
import numpy as np
import tensorflow as tf
import mlflow
import mlflow.tensorflow
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
seed = 87
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

x_train_small = x_train[:20000]
y_train_small = y_train[:20000]
x_test_small = x_test[:4000]
y_test_small = y_test[:4000]

x_train_mlp = x_train_small.astype('float32') / 255.0
x_test_mlp = x_test_small.astype('float32') / 255.0

x_train_pre = keras.applications.mobilenet_v2.preprocess_input(x_train_small.astype('float32'))
x_test_pre = keras.applications.mobilenet_v2.preprocess_input(x_test_small.astype('float32'))

## 2. Baseline MLP

A simple Multi-Layer Perceptron that flattens 32x32x3 images into a 3072-dimensional vector. By discarding spatial relationships between pixels, the MLP struggles to capture complex visual patterns. It serves as a high-speed, low-accuracy reference point to measure the performance gain of Transfer Learning.

In [0]:
mlp_model = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

mlp_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

mlp_model.summary()

In [0]:
mlflow.tensorflow.autolog(
    silent=True,
    registered_model_name='ai_ml_in_practice.neural_networks_silver.cifar_mlp_baseline'
    )

with mlflow.start_run(run_name='cifar_mlp_baseline') as run:
    t0 = time.time()
    history = mlp_model.fit(
        x_train_mlp,
        y_train_small,
        epochs=10,
        batch_size=32,
        validation_split=0.2,
        verbose=2
    )

    dt = time.time() - t0
    test_loss, test_acc = mlp_model.evaluate(x_test_mlp, y_test_small, verbose=0)
    
    # Log the test metrics
    mlflow.log_metrics({
        "train_time_s": dt,
        "test_accuracy": float(test_acc),
        "test_loss": float(test_loss)
    })
    
    # Accuracy & Loss Summary Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(history.history['accuracy'], label='Train Acc')
    ax1.plot(history.history['val_accuracy'], label='Val Acc')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    
    plt.tight_layout()
    
    mlflow.log_figure(fig, "training_metrics_summary.png")

print(f'Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}')

## 3. Transfer learning with MobileNetV2

MLflow tracks both runs to evaluate the trade-off between computational cost and model performance. While Transfer Learning is more CPU-intensive due to the 2x Upsampling, it achieves a massive accuracy leap from the very first epoch.

Key metrics to compare:
- **Accuracy Gain**: Transfer learning typically starts at 60%+ accuracy, doubling the MLP baseline.
- **Convergence Speed**: Requires fewer epochs to reach peak performance.
- **Resource Trade-off**: Higher per-step latency (CPU-bound) vs. significantly higher predictive power.

Transfer Learning Mechanics:
- **Frozen Backbone**: Setting base_model.trainable = False preserves ImageNet-learned weights and skips gradient computation for millions of parameters.
- **Specialized Preprocessing**: preprocess_input() aligns CIFAR-10 data with the original training distribution (range [−1,1]).
- **Global Pooling**: Converts spatial feature maps into a dense vector, efficiently summarizing the extracted patterns.
- **Custom Head**: A lightweight classifier trained specifically on the new 10-class problem.

Why freeze? 
- **Universal Features**: Early layers detect edges and textures common to all images.
- **Data Scarcity**: Prevents overfitting on the reduced (20k) CIFAR-10 subset.
- **Efficiency**: Training the entire network on CPU would be prohibitively slow.

When to freeze vs. fine-tune:
- **Freeze**: Task is very similar to ImageNet (e.g., object classification)
- **Fine-tune**: Task is different (e.g., medical imaging, satellite imagery)
- **Hybrid**: Freeze early layers, train later layers (layers learn more task-specific features)

In [0]:
base_model = keras.applications.MobileNetV2(
    input_shape=(64, 64, 3),
    include_top=False, 
    weights='imagenet'
)
base_model.trainable = False

transfer_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.UpSampling2D(size=(2, 2), interpolation='nearest'), 
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation='softmax')
])

transfer_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

transfer_model.summary()

In [0]:
mlflow.tensorflow.autolog(
    silent=True,
    registered_model_name='ai_ml_in_practice.neural_networks_silver.cifar_mobilenet_transfer'
    )

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=1,
    restore_best_weights=True,
    verbose=1
)

with mlflow.start_run(run_name='cifar_mobilenet_transfer'):
    t0 = time.time()
    history = transfer_model.fit(
        x_train_pre,
        y_train_small,
        epochs=10,
        batch_size=64,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=2
    )

    dt = time.time() - t0
    test_loss, test_acc = transfer_model.evaluate(x_test_pre, y_test_small, verbose=0)
    
    # Log the test metrics
    mlflow.log_metrics({
        "train_time_s": dt,
        "test_accuracy": float(test_acc),
        "test_loss": float(test_loss)
    })
    
    # Accuracy & Loss Summary Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(history.history['accuracy'], label='Train Acc')
    ax1.plot(history.history['val_accuracy'], label='Val Acc')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    
    plt.tight_layout()
    
    mlflow.log_figure(fig, "training_metrics_summary.png")

## 4. Models comparison

In the final stage, we move beyond global metrics (Accuracy/Loss) to inspect how the models actually "see" the data. We load the trained models from the MLflow registry and put them to the test on specific edge cases from the CIFAR-10 test set.

**Loading & Inference**  
We retrieve the MLP Baseline and the MobileNetV2 Transfer Model. By feeding them the same image, we can contrast how a "spatially blind" network compares to one that understands high-level visual features.

**The Ambiguity Tool**  
Our custom comparison function allows us to slice the test data in three ways:
- **Highest Confidence**: Displays images where the model is most certain. This usually shows the "cleanest" examples of a class.
- **Lowest Confidence (Ambiguity)**: This is the most revealing mode. It highlights images that confuse the model, showing where the decision boundaries are blurry.
- **Random**: Provides an unbiased look at general performance.

**Key Observation**  
By analyzing the Probability Distribution (Histograms), we can see if the model is "confidently wrong" or "honestly confused." A better model like MobileNetV2 will typically show higher, sharper peaks for the correct class, while the MLP often shows a flat, uncertain distribution across multiple unrelated categories.

In [0]:
mlp_model = mlflow.tensorflow.load_model(
    "models:/ai_ml_in_practice.neural_networks_silver.cifar_mlp_baseline/1"
)

transfer_model = mlflow.tensorflow.load_model(
    "models:/ai_ml_in_practice.neural_networks_silver.cifar_mobilenet_transfer/1"
)

In [0]:
def compare_models_on_image(mlp_model, transfer_model, x_mlp, x_trans, y_true, target_name='cat', mode='lowest'):
    """
    Compares MLP and MobileNetV2 on a specific image from CIFAR-10.
    target_name: Name of the class (e.g., 'cat', 'dog', 'truck')
    mode: 'lowest' (most ambiguous), 'highest' (most confident), 'random'.
    """
    # Map name to index
    cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
    try:
        target_class = cifar_classes.index(target_name.lower())
    except ValueError:
        print(f"Error: {target_name} is not a valid CIFAR-10 class.")
        return

    # Find indices of the target class
    indices = np.where(y_true == target_class)[0]
    
    # Get predictions to find the specific image based on mode
    probs_trans_all = transfer_model.predict(x_trans[indices], verbose=0)
    confidences = np.max(probs_trans_all, axis=1)
    
    if mode == 'lowest':
        rel_idx = np.argsort(confidences)[0]
    elif mode == 'highest':
        rel_idx = np.argsort(confidences)[-1]
    else: # random
        rel_idx = np.random.randint(0, len(indices))
        
    idx = indices[rel_idx]
    
    # Get final predictions for both models
    p_mlp = mlp_model.predict(x_mlp[idx:idx+1], verbose=0)[0]
    p_trans = transfer_model.predict(x_trans[idx:idx+1], verbose=0)[0]

    # Visualization
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))

    # Subplot 1: Image
    ax[0].imshow(x_mlp[idx])
    ax[0].set_title(f"Actual: {target_name.upper()}\nSelection: {mode.capitalize()}")
    ax[0].axis('off')

    # Subplot 2: MLP Results
    ax[1].bar(cifar_classes, p_mlp, color='gray', alpha=0.6)
    ax[1].bar(cifar_classes[np.argmax(p_mlp)], p_mlp[np.argmax(p_mlp)], color='tab:red')
    ax[1].set_title(f"MLP (Pred: {cifar_classes[np.argmax(p_mlp)]})")
    ax[1].set_xticklabels(cifar_classes, rotation=45)
    ax[1].set_ylim(0, 1.1)

    # Subplot 3: MobileNetV2 Results
    ax[2].bar(cifar_classes, p_trans, color='tab:blue', alpha=0.6)
    ax[2].bar(cifar_classes[np.argmax(p_trans)], p_trans[np.argmax(p_trans)], color='tab:green')
    ax[2].set_title(f"MobileNetV2 (Pred: {cifar_classes[np.argmax(p_trans)]})")
    ax[2].set_xticklabels(cifar_classes, rotation=45)
    ax[2].set_ylim(0, 1.1)

    plt.tight_layout()
    plt.show()

In [0]:
# Pick among CIFAR classes:
# cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

compare_models_on_image(
    mlp_model, 
    transfer_model, 
    x_test_mlp, 
    x_test_pre, 
    y_test_small, 
    target_name='horse', 
    mode='random'
)